# Numerical mixing in ACCESS-OM3 25 km: zonal means

How much of the temperature mixing is done by the advection scheme, compared with the vertical mixing scheme?

Temperature first, then the same for salinity at the end.

Run: `MC_25km_jra_iaf+wombatlite-test5.1`, monthly output, time mean over 2008–2018.

## Notation

Using different notation than mentioned elsewhere, χ_num (numerical) and χ_param (parameterised). In the plots and code
they are called $\chi^{adv}$ and $\chi^{mix}$, trying to go off J. Bisits's paper (equation numbers refer to the draft).
See also `numerical_mixing_ACCESS_OM3.ipynb`, which looks at $\chi^{adv}$ on its own.

- $\chi^{adv}$: variance production by the lateral advection scheme (paper eqs. 33–35).
  It leaves out the remapping term $\chi^{remap}$ (paper eq. 42).
- $\chi^{mix}$: variance production by parameterised vertical diffusion (paper eqs. 54–55).
  It leaves out neutral (along-isopycnal) diffusion, which test5.1 also runs.

Both are temperature variance production per unit area (°C² m s⁻¹). Negative means mixing.

Where they come from:

- $\chi^{adv}$ is online: the MOM6 diagnostic `T_advection_scheme_variance_production`, calculated every
  time step inside the model and used as it is.
- $\chi^{mix}$ is offline: calculated here from monthly means of `thetao`, `thkcello` and `difvho`, as
  $-2\kappa\,(\Delta\theta)^2/\Delta z$ at each interface, half given to the cell above and half to the cell below.

Not really a fair comparison...

## The numerical fraction F

$$F = \frac{\chi^{adv}}{\chi^{adv} + \chi^{mix}}$$

Both terms are negative where there is mixing, so the signs cancel and F is the share of the temperature variance
destruction that the advection scheme does.

- F = 0: no numerical mixing, the vertical mixing scheme does all of it.
- F = 0.5: the two are equal.
- F = 1: the advection scheme does all of it.
- F < 0 or F > 1: only where the advection scheme creates variance ($\chi^{adv} > 0$) instead of destroying it.

F can be high where there is very little mixing in total, so look at it together with the $\chi^{adv}$ and $\chi^{mix}$ panels.

For a region, the two terms are added up first (area weighted) and F is taken from the sums.


In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
# e.g. https://access-om3-configs.access-hive.org.au/latest/Experiments/
# esm_file = "/g/data/ol01/outputs/access-om3-8km/MC_8km_jra_ryf-test2-expt-c349f800/experiment_datastore.json"
# esm_file = "/g/data/ik11/outputs/access-om3-025/MC_25km_era_iaf+wombatlite-test5.1-era-b3d64e3d/datastore.json"
# esm_file = "/g/data/ik11/outputs/access-om3-025/MC_25km_jra_ryf+wombatlite-test5.1-5c6ad028/datastore.json"
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
if not papermill: 
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import os
from functools import partial
from glob import glob
from multiprocessing import get_context

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm, LogNorm, BoundaryNorm
import cmocean

## Model output

One file per variable per year, in one `output` folder per year.

In [ ]:
run = os.path.dirname(esm_file)      # the experiment folder, from the datastore path set at the top
years = range(2008, 2019)
n_workers = 6        # CPUs to use for the time mean. 1 works too, it just takes about 2 hours


def find(var, year):
    return glob(f'{run}/output*/access-om3.mom6.3d.{var}.1mon.mean.{year}.nc')[0]

In [ ]:
# grid, cell area and depth
with xr.open_dataset(f'{run}/output000/access-om3.mom6.static.nc') as static:
    deptho = static.deptho.load()
    areacello = static.areacello.load()

with xr.open_dataset(find('difvho', years[0]), decode_timedelta=False) as f:
    zi = f.zi.values                  # interface depths, 76 of them
with xr.open_dataset(find('thkcello', years[0]), decode_timedelta=False) as f:
    zl, yh, xh = f.zl.values, f.yh.values, f.xh.values

# ocean cells: the top of the layer is above the sea floor
ocean = zi[:-1, None, None] < deptho.fillna(0).values

## $\chi^{mix}$ for one month

At the interface between cell $k-1$ and cell $k$:

$$\chi^{mix}_{k-1/2} = -2\,\kappa_{k-1/2}\,\frac{(\theta_k-\theta_{k-1})^2}{\tfrac12(h_{k-1}+h_k)}$$

Half goes to the cell above and half to the cell below. Nothing at the surface, the sea floor or next to land.

`thetao` is on z levels and `thkcello` on the model's own layers. In z* these are the same to within 2 %, so they
are treated as the same layers.

In [ ]:
def chi_mix_month(T, h, kappa, wet):
    # T, h, wet are on the 75 layers, kappa on the 76 interfaces
    dT = T[1:] - T[:-1]
    dz = 0.5 * (h[:-1] + h[1:])
    k = kappa[1:-1]                                   # interfaces between two layers
    good = wet[:-1] & wet[1:] & np.isfinite(dT) & np.isfinite(k) & (dz > 0)
    at_interface = np.where(good, -2 * k * dT**2 / np.where(good, dz, 1), 0)

    chi = np.zeros_like(T)
    chi[:-1] += 0.5 * at_interface                    # cell above
    chi[1:] += 0.5 * at_interface                     # cell below
    return np.where(wet, chi, np.nan)

## Time mean, 2008–2018

$\chi^{mix}$ is calculated for each month and then averaged, together with $\chi^{adv}$, temperature
and layer thickness. You could probably use dask to make these calculations more efficient... 

In [ ]:
def time_mean(rows, tracer, kappa, adv):
    # tracer, kappa, adv are the variable names, e.g. 'thetao', 'difvho', 'T_advection_scheme_variance_production'
    y0, y1 = rows
    wet = ocean[:, y0:y1]
    total, count = {}, {}

    for year in years:
        files = {tracer: xr.open_dataset(find(tracer + '+z', year), decode_timedelta=False),
                 'thkcello': xr.open_dataset(find('thkcello', year), decode_timedelta=False),
                 kappa: xr.open_dataset(find(kappa, year), decode_timedelta=False),
                 adv: xr.open_dataset(find(adv, year), decode_timedelta=False)}

        for month in range(12):
            read = lambda var: files[var][var].isel(time=month, yh=slice(y0, y1)).values.astype('float64')
            c, h = read(tracer), read('thkcello')
            fields = {'tracer': c, 'h': h,
                      'chi_adv': np.where(wet, read(adv), np.nan),
                      'chi_mix': chi_mix_month(c, h, read(kappa), wet)}

            for name, field in fields.items():
                ok = np.isfinite(field)
                total[name] = total.get(name, 0) + np.where(ok, field, 0)
                count[name] = count.get(name, 0) + ok

        for f in files.values():
            f.close()

    return {name: (total[name] / np.where(count[name] > 0, count[name], np.nan)).astype('float32')
            for name in total}

In [ ]:
step = 192      # latitude rows per block (the files are chunked in 192 rows)
blocks = [(y0, min(y0 + step, len(yh))) for y0 in range(0, len(yh), step)]


def global_time_mean(tracer, kappa, adv):
    one_block = partial(time_mean, tracer=tracer, kappa=kappa, adv=adv)
    parts = []
    with get_context('fork').Pool(n_workers) as pool:
        for part in pool.imap(one_block, blocks):
            parts.append(part)
            print(f'block {len(parts)} of {len(blocks)} done')

    ds = xr.Dataset({name: (('zl', 'yh', 'xh'), np.concatenate([p[name] for p in parts], axis=1))
                     for name in parts[0]},
                    coords={'zl': zl, 'yh': yh, 'xh': xh})
    ds['deptho'] = (('yh', 'xh'), deptho.values)
    ds['areacello'] = (('yh', 'xh'), areacello.values)
    ds['ocean'] = (('zl', 'yh', 'xh'), ocean)
    return ds


ds = global_time_mean('thetao', 'difvho', 'T_advection_scheme_variance_production')
area = ds.areacello.where(ds.ocean, 0)     # cell area, zero on land
ds

## Basins

Simple longitude/latitude boxes for the Pacific, Atlantic and Indian oceans. Model longitudes run from -280 to 80.

In [ ]:
x, y = ds.xh, ds.yh
wet = ds.deptho > 0

# Pacific: 120E (147E south of 30S) to 70W, following Central America north of 8N
west = xr.where(y < -30, -213, -240)
east = xr.where(y < 8, -70, np.interp(y, [8, 10, 13, 15, 17, 21, 30, 66],
                                         [-77, -84, -88, -93, -100, -106, -115, -115]))
pacific = (x >= west) & (x <= east) & (y <= 65)

# Atlantic: 100W to 20E, without the Pacific, Mediterranean, Hudson Bay and Baltic
atlantic = (x >= -100) & (x <= 20) & (y <= 65) & ~pacific
atlantic = atlantic & ~((y > 30) & (y < 48) & (x > -6))
atlantic = atlantic & ~((y > 50) & (x < -70))
atlantic = atlantic & ~((y > 53) & (x > 12))

# Indian: 20E to 120E (147E south of 30S, 100E north of 8S to leave out the Indonesian seas), up to 30N,
# without the Red Sea and Persian Gulf. 80E to 120E is -280 to -240 in model longitudes.
east_indian = xr.where(y < -30, -213, xr.where(y < -8, -240, -260))
indian = ((x > 20) | (x < east_indian)) & (y <= 30)
indian = indian & ~((y > 12) & (x > 32) & (x < 44))
indian = indian & ~((y > 23.5) & (x > 47) & (x < 57))

basins = {'Global': wet, 'Pacific': pacific & wet, 'Atlantic': atlantic & wet, 'Indian': indian & wet}

# quick check of the masks
fig, ax = plt.subplots(figsize=(10, 4))
(pacific + 2 * atlantic + 3 * indian).where(wet).plot(
    ax=ax, x='xh', y='yh', levels=[-0.5, 0.5, 1.5, 2.5, 3.5],
    colors=['lightgrey', 'tab:blue', 'tab:orange', 'tab:green'], add_colorbar=False)
ax.set_title('Pacific (blue), Atlantic (orange) and Indian (green)')
plt.show()

## Zonal mean

Two ways of getting F along a latitude circle:

- **F**: add up $\chi^{adv}$ and $\chi^{mix}$ along longitude (area weighted), then take the ratio.
  This is the share of all the mixing at that latitude and depth that the advection scheme does.
- **F of a typical cell**: take the ratio in every cell, then average along longitude (volume weighted).
  Only cells with F between 0 and 1 are used, because F is not bounded where the scheme creates variance.
- In thermocline bias meeting we discussed F averaged at the end, but to interprete as a fraction between 0<F<1 I think it would be better to go with average first then find fraction.

In [ ]:
min_rate = 1e-12    # °C² s⁻¹. Less mixing than this is too little to say anything about F


def zonal_mean(ds, mask, min_rate):
    A = area.where(mask, 0)
    vol = ds.h * A                          # cell volume
    V = vol.sum('xh')

    adv = (ds.chi_adv * A).sum('xh')
    mix = (ds.chi_mix * A).sum('xh')

    out = xr.Dataset()
    out['chi_adv'] = adv / V                # per unit volume
    out['chi_mix'] = mix / V
    out['tracer'] = (ds.tracer * vol).sum('xh') / V
    out['F'] = adv / (adv + mix)

    # F in each cell first, then the volume weighted mean
    F_cell = ds.chi_adv / (ds.chi_adv + ds.chi_mix)
    rate = abs(ds.chi_adv + ds.chi_mix) / ds.h
    use = (F_cell >= 0) & (F_cell <= 1) & (rate > min_rate)
    w = vol.where(use, 0)
    out['F_cell'] = (F_cell.where(use, 0) * w).sum('xh') / w.sum('xh')

    return out.transpose('zl', 'yh')


zonal = {name: zonal_mean(ds, mask, min_rate) for name, mask in basins.items()}

## Plots

The depth axis is stretched, log(depth + 100 m), so the thermocline gets most of the room and the top 100 m is
not blown up the way it is on a plain log axis.

In the F panels white is no numerical mixing and darker red is a bigger share. Blue is where the advection scheme
creates variance instead of destroying it. Grey is land, or too little mixing to say.

In [ ]:
F_levels = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 1]
F_cmap = plt.get_cmap('cmo.amp', len(F_levels) - 1).copy()
F_cmap.set_under('#6a9bd8')
F_cmap.set_over('#0b1f4d')
F_norm = BoundaryNorm(F_levels, F_cmap.N)


def stretched_depth(ax):
    ax.set_yscale('function', functions=(lambda z: np.log10(np.maximum(z, 0) + 100),
                                         lambda s: 10**s - 100))
    ax.set_ylim(float(ds.zl.max()), 0)
    ax.set_yticks([0, 100, 200, 500, 1000, 2000, 4000])
    ax.set_ylabel('depth (m)')


def plot_zonal(z, name, units='°C² s⁻¹', vmax=1e-7, min_rate=min_rate, contours=np.arange(0, 31, 2)):
    lat, depth = z.yh, z.zl
    enough = abs(z.chi_adv + z.chi_mix) > min_rate

    fig, axes = plt.subplots(4, 1, figsize=(10, 13), sharex=True, constrained_layout=True)

    p = axes[0].pcolormesh(lat, depth, z.chi_adv, cmap='RdBu', norm=SymLogNorm(vmax * 1e-4, vmin=-vmax, vmax=vmax))
    fig.colorbar(p, ax=axes[0], label=units)
    axes[0].set_title('Advection scheme, $\\chi^{adv}$ (red = mixing)')

    p = axes[1].pcolormesh(lat, depth, -z.chi_mix, cmap='cmo.matter', norm=LogNorm(vmax * 1e-5, vmax))
    fig.colorbar(p, ax=axes[1], label=units)
    axes[1].set_title('Vertical mixing, $-\\chi^{mix}$')

    p = axes[2].pcolormesh(lat, depth, z.F.where(enough), cmap=F_cmap, norm=F_norm)
    fig.colorbar(p, ax=axes[2], label='F', extend='both', ticks=F_levels)

    p = axes[3].pcolormesh(lat, depth, z.F_cell.where(enough), cmap=F_cmap, norm=F_norm)
    fig.colorbar(p, ax=axes[3], label='F', extend='both', ticks=F_levels)

    axes[2].set_title(r'Numerical Fraction $F = \sum_x \chi^{adv} A \;/\; \sum_x (\chi^{adv} + \chi^{mix}) A$')
    axes[3].set_title(r'$\langle F_{cell} \rangle = \sum_x F_{cell} V \;/\; \sum_x V$,   '
                      r'$F_{cell} = \chi^{adv} / (\chi^{adv} + \chi^{mix})$,   $0 \leq F_{cell} \leq 1$')
    for ax in axes:
        ax.set_facecolor('grey')
        ax.contour(lat, depth, z.tracer, levels=contours, colors='k', linewidths=0.4)   # isotherms or isohalines
        stretched_depth(ax)
        ax.set_xlim(-75, 65)
    axes[-1].set_xlabel('latitude (°N)')
    fig.suptitle(f'{name} zonal mean, {os.path.basename(run)}, {years[0]}–{years[-1]}')
    return fig

In [ ]:
fig = plot_zonal(zonal['Global'], 'Global')
mkmd.savefig(fig, 'Numerical mixing fraction, temperature',
             'Global zonal mean of temperature variance production by the advection scheme and by vertical '
             'mixing, and the numerical fraction F.', dpi=150)

In [ ]:
fig = plot_zonal(zonal['Pacific'], 'Pacific')

In [ ]:
fig = plot_zonal(zonal['Atlantic'], 'Atlantic')

In [ ]:
fig = plot_zonal(zonal['Indian'], 'Indian')

## Salinity

The same thing for salinity.

- $\chi^{adv}$: the online diagnostic `S_advection_scheme_variance_production`.
- $\chi^{mix}$: offline again, from monthly `so`, `thkcello` and the salt diffusivity `difvso`, same formula.

Units are psu² s⁻¹. Salinity differences are much smaller numbers than temperature differences, so both terms
should be roughly 100 times smaller. The colour scales and the "too little mixing" cut-off are set 100 times smaller to match (change `vmax` if the plots look washed out).
F has no units, so it can be compared directly with the temperature F.

This reads another 11 years of output, so about 20 more minutes.

In [ ]:
ds_S = global_time_mean('so', 'difvso', 'S_advection_scheme_variance_production')

In [ ]:
min_rate_S = 1e-14      # psu² s⁻¹

zonal_S = {name: zonal_mean(ds_S, mask, min_rate_S) for name, mask in basins.items()}

Black lines are isohalines every 0.25 psu.

In [ ]:
figs_S = {name: plot_zonal(zonal_S[name], f'{name} salinity', units='psu² s⁻¹', vmax=1e-9, min_rate=min_rate_S,
                           contours=np.arange(33, 37.1, 0.25))
          for name in basins}
mkmd.savefig(figs_S['Global'], 'Numerical mixing fraction, salinity',
             'Global zonal mean of salinity variance production by the advection scheme and by vertical '
             'mixing, and the numerical fraction F.', dpi=150)